# Low-rank evidence adaptation
How many parameters need gradients?

In [ ]:
import os
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(root / 'src'))
import torch
torch.set_num_threads(1)
torch.manual_seed(42)
smoke = os.environ.get('PYTHONIC_SMOKE') == '1'

In [ ]:
from pythonic.model import Decoder, DecoderConfig
from pythonic.lora import inject_lora
model = Decoder(DecoderConfig(width=32, layers=1, context=32))
x = torch.randint(0, 258, (1, 16))
before = model(x).detach()
print('replaced:', inject_lora(model, rank=4))
print('trainable:', sum(p.numel() for p in model.parameters() if p.requires_grad))
print('total:', sum(p.numel() for p in model.parameters()))
torch.testing.assert_close(model(x), before)

In [ ]:
if not smoke:
    from pythonic.biomedical import load_biomedical, train_evidence
    from pythonic.data import load_papers, split_papers
    from pythonic.training import TrainConfig
    os.environ.setdefault('HF_HOME', str(root / 'data/hf'))
    split = split_papers(load_papers(root / 'data/pubmedqa.json'))
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model, tokenizer = load_biomedical(device)
    run = TrainConfig(steps=10, accumulation=1, device=device)
    report = train_evidence(model, tokenizer, split.train, split.validation, run, root / 'artifacts/notebook-evidence')
    print(report['validation'])
    print('peak allocated bytes:', report['peak_bytes'])
else:
    print('Pretrained training skipped in offline smoke mode.')

Zero-initialized output adapters preserve the base model initially. Only the low-rank matrices receive gradients. Reference: [LoRA](https://arxiv.org/abs/2106.09685).